# Warden pre-post toxicity check

This notebook checks a draft before it is posted. It does not rewrite the user's words. If the draft is likely toxic, it returns the category scores and asks the user to rethink the comment.

In [11]:
# Run once if needed:
# %pip install transformers torch

import os
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')

from pathlib import Path
import json
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

TOXICITY_PATH = Path('../models/toxic-bert-finetuned')
LABELS = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
TOXICITY_THRESHOLD = 0.40
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if not TOXICITY_PATH.exists():
    raise FileNotFoundError(f'Model not found: {TOXICITY_PATH.resolve()}')

toxicity_tokenizer = AutoTokenizer.from_pretrained(
    TOXICITY_PATH, local_files_only=True
)
toxicity_model = AutoModelForSequenceClassification.from_pretrained(
    TOXICITY_PATH, local_files_only=True
).to(DEVICE).eval()

print('device:', DEVICE)
print('toxicity threshold:', TOXICITY_THRESHOLD)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

device: cuda
toxicity threshold: 0.4


In [12]:
def score_draft(text: str) -> dict[str, float]:
    inputs = toxicity_tokenizer(
        text, return_tensors='pt', truncation=True, max_length=192
    )
    inputs = {key: value.to(DEVICE) for key, value in inputs.items()}
    with torch.inference_mode():
        probabilities = torch.sigmoid(toxicity_model(**inputs).logits[0]).cpu()
    return {label: round(float(probabilities[i]), 4) for i, label in enumerate(LABELS)}

def check_draft(text: str, toxicity_threshold: float = TOXICITY_THRESHOLD) -> dict:
    scores = score_draft(text)
    is_toxic = scores['toxic'] >= toxicity_threshold
    return {
        'text': text,
        'toxicity_score': scores['toxic'],
        'categories': scores,
        'is_toxic': is_toxic,
        'action': 'warn' if is_toxic else 'allow',
        'message': (
            'This comment may be toxic. Please rethink it before posting.'
            if is_toxic else
            'This comment is below the toxicity warning threshold.'
        ),
    }

In [13]:
draft = 'you are such a loser and nobody likes you.'
result = check_draft(draft)
print(json.dumps(result, indent=2))

{
  "text": "you are such a loser and nobody likes you.",
  "toxicity_score": 0.952,
  "categories": {
    "toxic": 0.952,
    "severe_toxic": 0.0031,
    "obscene": 0.0488,
    "threat": 0.0012,
    "insult": 0.8195,
    "identity_hate": 0.0067
  },
  "is_toxic": true,
  "action": "warn",
  "message": "This comment may be toxic. Please rethink it before posting."
}


The warning does not edit or automatically submit the comment. The user can revise it, cancel it, or continue according to the product's chosen policy.